# Performance considerations in `duckdb`

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    os.system("pip install -U duckdb")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

In [ ]:
import duckdb

## Download data

For the performance analysis, we need a bigger dataset.
We will use the [2015 Flight Delays and Cancellations](https://www.kaggle.com/datasets/usdot/flight-delays)
dataset from Kaggle.

First, you need to download the dataset:

In [ ]:
import requests
req = requests.get("https://www.kaggle.com/api/v1/datasets/download/usdot/flight-delays")
with open("flights.zip",'wb') as output_file:
    output_file.write(req.content)

`duckdb` can directly read ZIP files with a community
extension. You need to install the extension first
(only once), then it can be loaded:

In [ ]:
duckdb.sql("INSTALL zipfs FROM community; LOAD zipfs") 

If the extension does not work for you, you can 
extract he files manually or use the following code
to extract the files (adjust the names in the `duckdb`
statements appropriately):
```python
from zipfile import ZipFile
with ZipFile("flights.zip", 'r') as zip_ref:
    zip_ref.extractall(".")
```

The number of airlines is quite small:

In [ ]:
duckdb.sql("SELECT * FROM 'zip://flights.zip/airlines.csv'").pl()

However, the number of total flights is much larger:

In [ ]:
duckdb.sql("SELECT * FROM 'zip://flights.zip/flights.csv'").pl()

`5,819,078`  rows with `31` columns is a quite big dataset.
The `SELECT` took quite long as the ZIP file needs to be read,
unpacked and then the CSV file needs to be loaded.

To save time, ingest it into a local table first:

In [ ]:
duckdb.sql("CREATE OR REPLACE TABLE flights AS SELECT * FROM 'zip://flights.zip/flights.csv'")

If you take a look at the Kaggle page, you will see that there
are more CSV files inside the ZIP file, i.e. 
`airlines.csv` and `airports.csv`. We will also ingest these files
into tables:

In [ ]:
duckdb.sql("CREATE OR REPLACE TABLE airlines AS SELECT * FROM 'zip://flights.zip/airlines.csv'")
duckdb.sql("CREATE OR REPLACE TABLE airports AS SELECT * FROM 'zip://flights.zip/airports.csv'")

Take a look at the `flights` table above. Some columns
are at least *opionated*, we would like to have a real
`date` columns. The same is true for `SCHEDULED_DEPARTURE`,
`DEPARTURE_TIME`, `SCHEDULED_ARRIVAL` and `ARRIVAL_TIME`.

We need to transform the data to the `date` format.
However, there are some difficulties:
* Take a look at the first row. `DEPARTURE_TIME` was
  early, but this means that it was on a *different day*.
  This can be fixed by adding the `DEPARTURE_DELAY` to
  the departure time.
* For flights leaving late in the evening, the `SCHEDULED_ARRIVAL`
  might be on the next day. The fix here is more complicated.
  If `SCHEDULED_ARRIVAL` is earlier than `SCHEDULED_DEPARTURE`,
  departure, it is the next day. Adding again `ARRIVAL_DELAY`
  will give us the real time of arrival

It is often easier to perform this transformation in 
different steps. If we make mistakes, we have to repeat
the procedure. Therefore, it is better to either add
*new columns* or *create new tables*. Modifying existing
tables will lead to non-idempotence and make life much more
difficult.

Let's star by creating a `DATE` column and populating it:

In [ ]:
duckdb.sql("ALTER TABLE flights ADD COLUMN IF NOT EXISTS DATE DATE")
duckdb.sql("UPDATE flights SET DATE=concat(year, '-', month, '-', day)")
duckdb.sql("SELECT DATE FROM flights").pl()

Looks good, not let's tackle the departure time. This is more
complicated, as we need to include the time of day:

In [ ]:
duckdb.sql("ALTER TABLE flights ADD COLUMN IF NOT EXISTS SCHEDULED_DEPARTURE_TIME TIMESTAMP")
duckdb.sql("UPDATE flights SET SCHEDULED_DEPARTURE_TIME=concat(year, '-', month, '-', day, ' ', substring(SCHEDULED_DEPARTURE, 1, 2), \
                                 ':', substring(SCHEDULED_DEPARTURE, 3), ':00')")
duckdb.sql("SELECT SCHEDULED_DEPARTURE_TIME FROM flights").pl()

Perfect! Calculating the actual departure time is a bit easier,
we just add the delay in minutes to the scheduled time and check
the result:

In [ ]:
duckdb.sql("ALTER TABLE flights ADD COLUMN IF NOT EXISTS ACTUAL_DEPARTURE_TIME TIMESTAMP")
duckdb.sql("UPDATE flights SET ACTUAL_DEPARTURE_TIME=SCHEDULED_DEPARTURE_TIME + INTERVAL(DEPARTURE_DELAY) MINUTES")
duckdb.sql("SELECT SCHEDULED_DEPARTURE_TIME, ACTUAL_DEPARTURE_TIME FROM flights").pl()

Excellent, even our year rollover has worked as we wanted it to work.

The scheduled arrival time is most complicated, we start with the same
method as for the scheduled departure time. However, we need to be
careful because some flights might arrive on the next day. We can take
this into account by adding one day to the arrival time if the arrival
time is earlier than the departure time (no time travel so far!):

In [ ]:
duckdb.sql("ALTER TABLE flights ADD COLUMN IF NOT EXISTS SCHEDULED_ARRIVAL_TIME TIMESTAMP")
duckdb.sql("UPDATE flights SET SCHEDULED_ARRIVAL_TIME=concat(year, '-', month, '-', day, ' ', substring(SCHEDULED_ARRIVAL, 1, 2), \
                                 ':', substring(SCHEDULED_ARRIVAL, 3), ':00')")
duckdb.sql("UPDATE flights SET SCHEDULED_ARRIVAL_TIME=SCHEDULED_ARRIVAL_TIME + INTERVAL 24 HOURS \
                   WHERE SCHEDULED_ARRIVAL_TIME<SCHEDULED_DEPARTURE_TIME")
duckdb.sql("SELECT SCHEDULED_ARRIVAL_TIME FROM flights").pl()

As you can see in the last few lines, also this year rollover worked
as expected!

Calculating the actual arrival time is now simpler, we just add the
delay and check the results:

In [ ]:
duckdb.sql("ALTER TABLE flights ADD COLUMN IF NOT EXISTS ACTUAL_ARRIVAL_TIME TIMESTAMP")
duckdb.sql("UPDATE flights SET ACTUAL_ARRIVAL_TIME=SCHEDULED_ARRIVAL_TIME + INTERVAL(ARRIVAL_DELAY) MINUTES")
duckdb.sql("SELECT SCHEDULED_ARRIVAL_TIME, ACTUAL_ARRIVAL_TIME FROM flights").pl()

Although this might look very *tedious*, you will often
encounter exactly these transformation in your data ingest.
Now we have a table which is (hopefully) *clean*.

At this point, it's a good idea to save this in a parquet
file. As we want to be able to check results or make corrections
at a later stage, we also save the original data. Depending
on the size of your data set, this is not always a good idea.
It could happen that it is much better to just save the fields
you need!

In [ ]:
%%time
duckdb.sql("COPY (FROM FLIGHTS) to 'flights.parquet'")

You can see that `duckdb` uses not just a single CPU, but
many. User time is much longer than wall time.

Reading from (especially spinning) disks is slow. Therefore,
it can be a good idea to save the files in maximum compression.
This takes much more time.

Of course, decompression is also slower. But normally, this is
easily compensated by the much shorter time which you need to
read the data from disk.

Try to increase the compression level:

In [ ]:
%%time
duckdb.sql("COPY (FROM FLIGHTS) to 'flights-l22.parquet'\
            (FORMAT 'PARQUET', CODEC  'zstd', COMPRESSION_LEVEL 22, ROW_GROUP_SIZE 100000)")

The actual waiting time has not increased tremendously,
but the CPU time is much longer. How is this reflected
in the file size?

In [ ]:
!ls -l flights*.parquet

In this case, we only save 15%. Especially if you also have text
in your data, the savings are much larger (more like 30%).

## Aggregations

Let's see how `duckdb` works on the `parquet` file and on the
internal data for simple aggregations. For this, we check
the average departure and arrival delay on certain routes:

In [ ]:
duckdb.sql("SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            FROM flights GROUP BY ALL ORDER BY ARRIVAL_DELAY").pl()

There are some *strange* airports here. Where do they come from? For this,
count how often they appear:

In [ ]:
duckdb.sql("SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            count(*) AS count \
            FROM flights GROUP BY ALL ORDER BY ARRIVAL_DELAY").pl()

Obviously, these are *minor airports* without an IATA code. To remove them,
we just consider airports with three-letter acronyms:

In [ ]:
duckdb.sql("SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            count(*) AS count \
            FROM flights \
            WHERE len(ORIGIN_AIRPORT)=3 AND len(DESTINATION_AIRPORT)=3\
            GROUP BY ALL ORDER BY ARRIVAL_DELAY").pl()

Much better. Some flight routes are not very popular. If we were serious
data scientists, we would probably remove those with too small sample size.
But we are rather interested in the performance.

Therefore, we analyze the statement via `EXPLAIN`. Unfortunately, this does not display
correctly in a `DataFrame`, therefore we need to work around it and 
`print` it:

In [ ]:
print(duckdb.sql("EXPLAIN SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            count(*) AS count \
            FROM flights \
            WHERE len(ORIGIN_AIRPORT)=3 AND len(DESTINATION_AIRPORT)=3\
            GROUP BY ALL ORDER BY ARRIVAL_DELAY").fetchone()[1])

If you want more complete information, you can use
`EXPLAIN ANALYZE`. Compared to pure `EXPLAIN`, this 
does not just show you the query plan, but also the
time it takes to execute it:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            count(*) AS count \
            FROM flights \
            WHERE len(ORIGIN_AIRPORT)=3 AND len(DESTINATION_AIRPORT)=3\
            GROUP BY ALL ORDER BY ARRIVAL_DELAY").fetchone()[1])

Compare this to what happens if we use the `parquet`
file instead:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
            avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
            count(*) AS count \
            FROM 'flights-l22.parquet' \
            WHERE len(ORIGIN_AIRPORT)=3 AND len(DESTINATION_AIRPORT)=3\
            GROUP BY ALL ORDER BY ARRIVAL_DELAY").fetchone()[1])

The total time is slower, but not by far. In many cases,
you do not necessarily need `duckdb`'s native format.
Saving `parquet` file give you additional interoperability
and long-time storage safety.

## Work with `JOIN`

If you remember the Kaggle page, there were several CSV files hosted there.
We have not yet made use of `airports.csv` which gives us the IATA codes
together with the corresponding airport names. Using that table, we will
also eliminate the airports with just numbers as they are not contained
in `airports.csv`. Of course, this only works if we use a standard join
and not an outer join:

In [ ]:
duckdb.sql("SELECT oa.AIRPORT AS origin, oa.STATE AS origin_state, \
                   da.AIRPORT AS destination, da.STATE AS destination_state,\
                   flights.* \
                   FROM flights, airports oa, airports da\
                   WHERE flights.ORIGIN_AIRPORT=oa.IATA_CODE AND flights.DESTINATION_AIRPORT=da.IATA_CODE").pl()

Let's analyze this query:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE SELECT oa.AIRPORT AS origin, oa.STATE AS origin_state, \
                   da.AIRPORT AS destination, da.STATE AS destination_state,\
                   flights.* \
                   FROM flights, airports oa, airports da\
                   WHERE flights.ORIGIN_AIRPORT=oa.IATA_CODE AND flights.DESTINATION_AIRPORT=da.IATA_CODE").fetchone()[1])

This takes much longer. Where does this come from? The reason is
that we are selecting all columns. This means, that the whole
`parquet` file needs to be read eliminating a lot of the
advantages of this columnar storage format.

Observe how this changes if we only select a single columns from
the (large) flights table:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE SELECT oa.AIRPORT AS origin, oa.STATE AS origin_state, \
                   da.AIRPORT AS destination, da.STATE AS destination_state,\
                   flights.ARRIVAL_DELAY \
                   FROM flights, airports oa, airports da\
                   WHERE flights.ORIGIN_AIRPORT=oa.IATA_CODE AND flights.DESTINATION_AIRPORT=da.IATA_CODE").fetchone()[1])

Let's try this again with the `parquet` file!
In this case, we use a CTE to keep the join
as simple as possible:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE \
                   WITH fl AS (SELECT * FROM 'flights-l22.parquet'), \
                        ai AS (SELECT * FROM 'zip://flights.zip/airports.csv') \
                   SELECT oa.AIRPORT AS origin, oa.STATE AS origin_state, \
                   da.AIRPORT AS destination, da.STATE AS destination_state,\
                   fl.ARRIVAL_DELAY \
                   FROM fl, ai oa, ai da\
                   WHERE fl.ORIGIN_AIRPORT=oa.IATA_CODE AND fl.DESTINATION_AIRPORT=da.IATA_CODE").fetchone()[1])

Again, the difference is not really big (30%). Obviously,
`parquet` files are already highly optimized for our
use case so that `duckdb` cannot speed up even if it uses
its internal format.

As a final example, take a look at a more complicated join
together with some aggregations:

In [ ]:
print(duckdb.sql("EXPLAIN ANALYZE\
                    WITH delays AS (SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, \
                       avg(DEPARTURE_DELAY) AS DEPARTURE_DELAY, avg(ARRIVAL_DELAY) AS ARRIVAL_DELAY,\
                       count(*) AS count \
                       FROM flights \
                       GROUP BY ALL ORDER BY ARRIVAL_DELAY)\
                     SELECT oa.AIRPORT AS origin, da.AIRPORT AS destination, DEPARTURE_DELAY, ARRIVAL_DELAY, count\
                            FROM delays, airports oa, airports da\
                            WHERE delays.ORIGIN_AIRPORT=oa.IATA_CODE AND delays.DESTINATION_AIRPORT=da.IATA_CODE").fetchone()[1])